<a href="https://colab.research.google.com/github/zain4cs/ML_Pipelines/blob/main/Titanic_using_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [50]:
import pandas as pd
import numpy as np

In [51]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_selection import SelectKBest, chi2

In [52]:
df = pd.read_csv("/content/drive/MyDrive/Datasets/train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [53]:
df.isnull().sum().sort_values(ascending=False)

,0
Cabin,687
Age,177
Embarked,2
PassengerId,0
Name,0
Pclass,0
Survived,0
Sex,0
Parch,0
SibSp,0


__________

**Let's Plan**

In [54]:
df.drop(columns=['PassengerId', 'Name', 'Ticket','Cabin'], inplace=True)

In [55]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['Survived']), df['Survived'], test_size=0.2, random_state=42)

In [56]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [57]:
  # Impute Missing Values

trf1 = ColumnTransformer([
      ('impute_age', SimpleImputer(), [2]),
      ('impute_embarked', SimpleImputer(strategy='most_frequent'), [-1])
  ], remainder='passthrough')

In [58]:
# Apply One-Hot-Encoder

trf2 = ColumnTransformer([
    ('ohe_sex', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), [1]),
    ('ohe_Embarked', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), [-1])
], remainder='passthrough')

In [59]:
#Use MinMax Scaling method Besause working on Feature!

trf3 = ColumnTransformer(([
    ('scale', MinMaxScaler(), slice(0,10))
]))

In [60]:
# Feature Selection!

trf4 = SelectKBest(score_func=chi2, k=8)

In [61]:
# Train Model

trf5 = DecisionTreeClassifier()

____________________________

**Create Pipeline**

In [62]:
pip = Pipeline([
    ('trf1', trf1),
    ('trf2', trf2),
    ('trf3', trf3),
    ('trf4', trf4),
    ('trf5', trf5)
])

**Pipeline Vs Make_Pipeline:**

Pipeline requires Naming of steps and Make_Pipeline Does Not require.

---



In [63]:
# pip = make_pipeline(trf1, trf2, trf3, trf4, trf5)

In [64]:
pip.fit(X_train, y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [-1])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1]),
                                                 ('ohe_Embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [-1])])),
                ('trf3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trf4',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x7857575d6660>)),
                ('trf5', DecisionTreeClassifier())])

In [66]:
pip.named_steps

{'trf1': ColumnTransformer(remainder='passthrough',
                   transformers=[('impute_age', SimpleImputer(), [2]),
                                 ('impute_embarked',
                                  SimpleImputer(strategy='most_frequent'),
                                  [-1])]),
 'trf2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_sex',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [1]),
                                 ('ohe_Embarked',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [-1])]),
 'trf3': ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 10, None))]),
 'trf4': SelectKBest(k=8, score_func=<function chi2 at 0x7857575d6660>),
 'trf5': DecisionTreeClassi